In [ ]:
import os
print(os.listdir())

['.config', 'sample_data']


In [15]:
import pandas as pd
import numpy as np
from datetime import datetime

orders = pd.read_csv('/content/olist_orders_dataset.csv')
order_items = pd.read_csv('/content/olist_order_items_dataset.csv')
customers = pd.read_csv('/content/olist_customers_dataset.csv')
products = pd.read_csv('/content/olist_products_dataset.csv')
sellers = pd.read_csv('/content/olist_sellers_dataset.csv')
order_payments = pd.read_csv('/content/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('/content/olist_order_reviews_dataset.csv')
product_category = pd.read_csv('/content/product_category_name_translation.csv')

print("All files loaded successfully!")
print("Orders shape:", orders.shape)

All files loaded successfully!
Orders shape: (99441, 8)


In [16]:
date_columns = ['order_purchase_timestamp', 'order_approved_at',
                'order_delivered_carrier_date', 'order_delivered_customer_date',
                'order_estimated_delivery_date']

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print("Date columns converted successfully!")
print(orders[date_columns].dtypes)

Date columns converted successfully!
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [ ]:
# Feature Engineering: Compute fulfillment duration & delay metrics
# Total lead time in days (Purchase to Customer Delivery)
orders['lead_time'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days

# Variance against target SLA date (Positive values denote late arrivals)
orders['delivery_delay'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

# Target flag for late deliveries
orders['is_delayed'] = orders['delivery_delay'].apply(lambda x: 1 if x > 0 else 0)

print("New features created:")
print(orders[['lead_time', 'delivery_delay', 'is_delayed']].head(10))

New features created:
   lead_time  delivery_delay  is_delayed
0        8.0            -8.0           0
1       13.0            -6.0           0
2        9.0           -18.0           0
3       13.0           -13.0           0
4        2.0           -10.0           0
5       16.0            -6.0           0
6        NaN             NaN           0
7        9.0           -12.0           0
8        9.0           -32.0           0
9       18.0            -7.0           0


In [ ]:
# Filter dataset for completed orders to maintain metric integrity
orders = orders[orders['order_status'] == 'delivered']

# Exclude records missing fulfillment timestamps
orders = orders.dropna(subset=['lead_time', 'delivery_delay'])

print("Orders after cleaning:", orders.shape)
print("Average Lead Time (days):", round(orders['lead_time'].mean(), 2))
print("Percentage of Delayed Orders: {:.2f}%".format(orders['is_delayed'].mean() * 100))

Orders after cleaning: (96470, 11)
Average Lead Time (days): 12.09
Percentage of Delayed Orders: 6.77%


In [ ]:
# Merge product category translation
products = products.merge(product_category, on='product_category_name', how='left')

order_items_clean = order_items[['order_id', 'product_id', 'seller_id', 'price', 'freight_value']]

print("Order items ready:", order_items_clean.shape)

Order items ready: (112650, 5)


In [ ]:
# Merging everything into one main table
df = orders.merge(order_items_clean, on='order_id', how='left')
df = df.merge(products[['product_id', 'product_category_name_english']], on='product_id', how='left')
df = df.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
df = df.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')

print("Master dataset created!")
print("Shape:", df.shape)
print(df.columns.tolist())

Master dataset created!
Shape: (110189, 18)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'lead_time', 'delivery_delay', 'is_delayed', 'product_id', 'seller_id', 'price', 'freight_value', 'product_category_name_english', 'customer_state', 'seller_state']


In [ ]:
# Target feature selection for dashboard visualization
columns_to_keep = [
    'order_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'lead_time',
    'delivery_delay',
    'is_delayed',
    'customer_state',
    'seller_state',
    'product_category_name_english',
    'price',
    'freight_value'
]

df_clean = df[columns_to_keep].copy()

# Calculate total transaction value per line item
df_clean['total_value'] = df_clean['price'] + df_clean['freight_value']

print("Clean dataset shape:", df_clean.shape)
print(df_clean.head())

Clean dataset shape: (110189, 13)
                           order_id order_purchase_timestamp  \
0  e481f51cbdc54678b7cc49136f2d6af7      2017-10-02 10:56:33   
1  53cdb2fc8bc7dce0b6741e2150273451      2018-07-24 20:41:37   
2  47770eb9100c2d0c44946d9cf07ec65d      2018-08-08 08:38:49   
3  949d5b44dbf5de918fe9c16f97b45f8a      2017-11-18 19:28:06   
4  ad21c59c0840e6cb83a9ceb5573f8159      2018-02-13 21:18:39   

  order_delivered_customer_date order_estimated_delivery_date  lead_time  \
0           2017-10-10 21:25:13                    2017-10-18        8.0   
1           2018-08-07 15:27:45                    2018-08-13       13.0   
2           2018-08-17 18:06:29                    2018-09-04        9.0   
3           2017-12-02 00:28:42                    2017-12-15       13.0   
4           2018-02-16 18:17:02                    2018-02-26        2.0   

   delivery_delay  is_delayed customer_state seller_state  \
0            -8.0           0             SP           SP   
1 

In [ ]:
# Perform final data quality check and summary profiling
print("Missing values:")
print(df_clean.isnull().sum())

print("\nBasic Statistics:")
print(df_clean[['lead_time', 'delivery_delay', 'total_value']].describe())

Missing values:
order_id                            0
order_purchase_timestamp            0
order_delivered_customer_date       0
order_estimated_delivery_date       0
lead_time                           0
delivery_delay                      0
is_delayed                          0
customer_state                      0
seller_state                        0
product_category_name_english    1559
price                               0
freight_value                       0
total_value                         0
dtype: int64

Basic Statistics:
           lead_time  delivery_delay    total_value
count  110189.000000   110189.000000  110189.000000
mean       12.007342      -12.029041     139.926806
std         9.451153       10.158194     189.324554
min         0.000000     -147.000000       6.080000
25%         6.000000      -17.000000      55.180000
50%        10.000000      -13.000000      92.120000
75%        15.000000       -7.000000     157.470000
max       209.000000      188.000000    69

In [ ]:
# Export staging file for dashboard consumption (e.g., Tableau, Power BI, Streamlit)
df_clean.to_csv('olist_supply_chain_cleaned.csv', index=False)
print("Cleaned file saved successfully as 'olist_supply_chain_cleaned.csv'")

Cleaned file saved successfully as 'olist_supply_chain_cleaned.csv'
